# Main Fig 2 — AUROC vs K (iso-compute)

**Data**: `heatmap_df_test.csv` per task/head from `final_results/phase0_v3/inference/`  
**Tasks**: main 5  
**Head**: Transformer  
**Layout**: 2 rows × 3 cols (5 panels, last cell hidden)

To change: `HEAD`, `TASKS`, `N_COLS`, or `METRIC`.

In [ ]:
%matplotlib inline

In [ ]:
import sys
from pathlib import Path

# ── Workspace root: auto-detect by looking for final_results/ ─────────────────
def _find_workspace():
    """Walk up from CWD until we find a directory containing final_results/."""
    candidate = Path.cwd().resolve()
    for _ in range(10):
        if (candidate / "final_results").exists():
            return candidate
        if candidate.parent == candidate:
            break
        candidate = candidate.parent
    # Explicit fallback (edit this if auto-detect fails)
    return Path("/Users/boshra/NSRR-workspace").resolve()

WORKSPACE_ROOT = _find_workspace()
NSRR_TOOLS     = WORKSPACE_ROOT / "NSRR-tools"
FINAL_RESULTS  = WORKSPACE_ROOT / "final_results"
PAPER_FIGURES  = NSRR_TOOLS / "results" / "paper_figures"
FINAL_OUT      = PAPER_FIGURES / "final"
FINAL_OUT.mkdir(parents=True, exist_ok=True)

# Add utils to path (notebooks/utils/)
_nb_dir = PAPER_FIGURES / "notebooks"
sys.path.insert(0, str(_nb_dir))

from utils.style import (
    apply_tbme_style, save_figure, FULL_W, HALF_W,
    MAIN_TASKS, SUPP_TASKS, ALL_TASKS, BINARY_MAIN,
    HEAD_STYLE, TASK_LABEL, FONT_ANNOT, FONT_BASE, FONT_LABEL,
)
from utils.data import set_root, load_analysis, load_heatmap, load_parquets
from utils import panels

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

set_root(WORKSPACE_ROOT)
apply_tbme_style()

# ── Confirm the workspace root is correct ─────────────────────────────────────
_ok = (WORKSPACE_ROOT / "final_results").exists()
print(f"WORKSPACE_ROOT : {WORKSPACE_ROOT}")
print(f"final_results/ : {'✓ found' if _ok else '✗ NOT FOUND — edit _find_workspace() fallback'}")

In [ ]:
# Panel labeling helper
def add_panel_label(ax, label, x=0.02, y=0.97):
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=8, fontweight="bold", va="top", fontfamily="serif")

In [ ]:
HEAD   = "transformer"
TASKS  = MAIN_TASKS          # change to ALL_TASKS for supplementary variant
METRIC = "auroc"
N_COLS = 3
N_ROWS = (len(TASKS) + N_COLS - 1) // N_COLS

# Load heatmap DataFrames
hmaps = {t: load_heatmap("phase0_v3", t, HEAD) for t in TASKS}
print({t: len(v) for t, v in hmaps.items()})

In [ ]:
# ROW_H controls how tall each row is (inches).
# Increase it to make panels bigger — try 2.5 or 3.0.
ROW_H = 2.1

# Build a subplot_mosaic layout so the last (incomplete) row is centred.
# Each panel label is repeated across 2 virtual columns; "." = ignored cell.
#
# Example for 5 tasks, N_COLS=3 (6 virtual columns):
#   a a b b c c
#   . d d e e .
labels  = [chr(97 + i) for i in range(len(TASKS))]
n_last  = len(TASKS) % N_COLS or N_COLS   # panels in the last row
n_full  = len(TASKS) // N_COLS

mosaic = []
for row in range(n_full):
    row_labels = labels[row * N_COLS : (row + 1) * N_COLS]
    mosaic.append([lbl for lbl in row_labels for _ in range(2)])

if n_last < N_COLS:   # incomplete → centre it
    last_labels = labels[n_full * N_COLS:]
    pad = N_COLS - n_last          # ignored cells on each side
    mosaic.append(
        ["."] * pad +
        [lbl for lbl in last_labels for _ in range(2)] +
        ["."] * pad
    )

fig, axd = plt.subplot_mosaic(mosaic, figsize=(FULL_W, N_ROWS * ROW_H))

for i, (lbl, task) in enumerate(zip(labels, TASKS)):
    ax = axd[lbl]
    panels.kvsk_panel(ax, hmaps[task], col=METRIC)
    ax.set_title(TASK_LABEL[task], fontsize=8)
    add_panel_label(ax, f"({chr(97+i)})")

fig.tight_layout(h_pad=1.5, w_pad=1.0)
plt.show()

In [ ]:
# ── Run when figure looks good ──────────────────────────────────
save_figure(fig, FINAL_OUT, "main_fig2_kvsk")
print("Saved →", FINAL_OUT / "main_fig2_kvsk.pdf")